<a href="https://colab.research.google.com/github/tariq-alhazmi/project_AI3101/blob/main/project_AI3101.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **استدعاء البيانات**

In [1]:
import pandas as pd
df = pd.read_csv('مدارس_المملكة_تقييم_سنوي_شامل.csv')
display(df.head())

,المنطقة,الإدارة,اسم المدرسة,قدرات_1443,تحصيلي_1443,التقويم_الذاتي_1443,التقويم_الخارجي_1443,الفرق_بين_التقويمين_1443,قدرات_1444,تحصيلي_1444,...,التقويم_الخارجي_1446,الفرق_بين_التقويمين_1446,قدرات_1447,تحصيلي_1447,التقويم_الذاتي_1447,التقويم_الخارجي_1447,الفرق_بين_التقويمين_1447,نسبة تقدم القدرات (%),نسبة تقدم التحصيلي (%),تقييم هيئة التقويم والتعليم
0,الباحة,إدارة التعليم بمحافظة المخواة,الثانوية الاولى بالمخواة نظام المقررات,72.88,67.06,78.96,64.83,14.13,73.91,72.76,...,68.72,8.23,70.96,71.48,82.44,67.27,15.17,-2.63%,6.59%,متقدم
1,الباحة,الإدارة العامة للتعليم بمنطقة الباحة,ثانوية أروى بنت عبد المطلب بن هشام بربوع الصف...,65.26,64.90,71.53,56.62,14.91,70.30,61.07,...,65.29,6.06,66.25,67.15,72.89,60.47,12.42,1.52%,3.47%,منطلق
2,الباحة,الإدارة العامة للتعليم بمنطقة الباحة,ثانوية أم عبد الله بن عمر بن الخطاب بالجبل وا...,74.70,73.64,83.76,69.05,14.71,72.74,76.09,...,67.17,2.06,72.61,75.17,83.95,71.34,12.61,-2.8%,2.08%,متقدم
3,الباحة,الإدارة العامة للتعليم بمنطقة الباحة,ثانوية أم عمارة بالمندق - مقررات,68.72,71.90,83.40,63.13,20.27,73.38,70.19,...,69.98,8.28,78.83,66.12,73.88,67.20,6.68,14.71%,-8.04%,متقدم
4,الباحة,الإدارة العامة للتعليم بمنطقة الباحة,ثانوية أم هشام بنت حارثة بالأطاولة - مقررات,71.69,71.28,77.55,73.60,3.95,67.03,73.46,...,69.12,12.57,68.66,76.76,76.96,69.58,7.38,-4.23%,7.69%,متقدم


# **Data Formatting & Handling Missing Values**

In [6]:
# 1 حذف المدارس ذات البيانات الناقصة
df_cleaned = df.dropna()
df_cleaned = df.dropna().copy()
# 2. إزالة علامة % وتحويل القيم إلى أرقام
columns_to_clean = ['نسبة تقدم القدرات (%)', 'نسبة تقدم التحصيلي (%)']
for col in columns_to_clean:
    if df_cleaned[col].dtype == 'object':
        df_cleaned[col] = df_cleaned[col].str.replace('%', '').astype(float)

display(df_cleaned[columns_to_clean].head())


,نسبة تقدم القدرات (%),نسبة تقدم التحصيلي (%)
0,-2.63,6.59
1,1.52,3.47
2,-2.80,2.08
3,14.71,-8.04
4,-4.23,7.69


# **Normalization**

In [7]:
from sklearn.preprocessing import MinMaxScaler

# تحديد الأعمدة التي تمثل المتغيرات المستقلة (سنستخدم بيانات 1447 كمثال للتدريب)
features = ['قدرات_1447', 'تحصيلي_1447', 'التقويم_الخارجي_1447', 'التقويم_الذاتي_1447']

# استدعاء أداة التقييس
scaler = MinMaxScaler()

# تطبيق التقييس على البيانات المحددة
df_cleaned[features] = scaler.fit_transform(df_cleaned[features])

# عرض أول 5 صفوف للتأكد من أن الأرقام تم ضغطها لتصبح بين 0 و 1
display(df_cleaned[features].head())

,قدرات_1447,تحصيلي_1447,التقويم_الخارجي_1447,التقويم_الذاتي_1447
0,0.542109,0.572114,0.521900,0.657232
1,0.452769,0.473143,0.382069,0.470818
2,0.573407,0.656457,0.605593,0.686707
3,0.691388,0.449600,0.520461,0.490142
4,0.498483,0.692800,0.569402,0.550264


# **(Linear Regression و Random Forest)**

In [9]:
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor

# 1. تحديد المتغيرات المستقلة (X) والتابع (Y)
#  المتغير التابع هو التحصيلي، والمستقلة هي بقية العوامل
X = df_cleaned[['قدرات_1447', 'التقويم_الخارجي_1447', 'التقويم_الذاتي_1447']]
y = df_cleaned['تحصيلي_1447']

# 2. تقسيم البيانات إلى مجموعة تدريب ومجموعة اختبار
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# 3. (Multiple Linear Regression) تدريب نموذج
model_lr = LinearRegression()
model_lr.fit(X_train, y_train)

print("--- (Multiple Linear Regression) نتائج نموذج  ---")
print(f"نقطة التقاطع (w0): {model_lr.intercept_:.4f}")
print(f"وزن القدرات (w1): {model_lr.coef_[0]:.4f}")
print(f"وزن التقويم الخارجي (w2): {model_lr.coef_[1]:.4f}")
print(f"وزن التقويم الذاتي (w3): {model_lr.coef_[2]:.4f}")

# 4. (Random Forest)تدريب نموذج
model_rf = RandomForestRegressor(random_state=42)
model_rf.fit(X_train, y_train)

print("\n--- (Random Forest) أهمية العوامل في  ---")
for feature, importance in zip(X.columns, model_rf.feature_importances_):
    print(f"{feature}: {importance:.4f}")

--- (Multiple Linear Regression) نتائج نموذج  ---
نقطة التقاطع (w0): 0.0505
وزن القدرات (w1): -0.4194
وزن التقويم الخارجي (w2): 0.7999
وزن التقويم الذاتي (w3): 0.4827

--- (Random Forest) أهمية العوامل في  ---
قدرات_1447: 0.1405
التقويم_الخارجي_1447: 0.6787
التقويم_الذاتي_1447: 0.1807
